In [1]:
column_types = {
"osm_id": "str",
"addr_street": "str",
"addr_number": "str",
"grupa_parter": "str",
"funkcja_parter": "str",
"grupa_pietra": "str",
"funkcja_pietra": "str",
"liczba_kondygnacji": "Int64",
"mieszana": "boolean",
"rok_budowy_szac": "Int64",
"styl_arch": "str",
"stan_tech": "Int64",
"wysokosc_max": "float64",
"zabytek": "str"
}

In [2]:
import pandas as pd
import numpy as np
import re
data_gdynia = pd.read_csv("../data/raw/srodmiescie_raw.csv",
                          sep=";", index_col = 0, dtype = column_types)
data_clean = data_gdynia.drop(columns='Unnamed: 18')
data_clean.head()

,osm_id,addr_street,addr_number,grupa_parter,funkcja_parter,grupa_pietra,funkcja_pietra,liczba_kondygnacji,mieszana,rok_budowy_szac,styl_arch,stan_tech,wysokosc_max,zabytek,zrodlo,pewnosc,data_sprawdzenia
id,,,,,,,,,,,,,,,,,
1,127184291,Świętego Piotra,8,NaN,Hotel,NaN,Hotel,7,<NA>,2000,Współczesność,3,NaN,NaN,google.com/maps,3.0,27.09.2026
2,890729731,Świętego Piotra,6D,NaN,Mieszkaniowa Wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,5,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3.0,27.09.2026
3,890292040,Węglowa,22,NaN,Centrum Handlowe,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3.0,27.09.2026
4,890729732,Świętego Piotra,6,NaN,Mieszkaniowa Wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3.0,27.09.2026
5,136885205,Tadeusza Wendy,15,NaN,Centrum Medyczne,NaN,Centrum Medyczne,6,<NA>,2000,Współczesność,3,NaN,NaN,google.com/maps,2.0,27.09.2026


In [3]:
data_clean = data_clean.dropna(thresh = 3)
data_clean.isna().sum()

osm_id                   0
addr_street            268
addr_number            310
grupa_parter          1091
funkcja_parter           0
grupa_pietra          1105
funkcja_pietra         437
liczba_kondygnacji       0
mieszana              1105
rok_budowy_szac          0
styl_arch                0
stan_tech                0
wysokosc_max          1105
zabytek               1105
zrodlo                   0
pewnosc                  0
data_sprawdzenia         0
dtype: int64

In [4]:
data_clean["data_sprawdzenia"] =pd.to_datetime(data_clean["data_sprawdzenia"], format = '%d.%m.%Y')
data_clean.head()

,osm_id,addr_street,addr_number,grupa_parter,funkcja_parter,grupa_pietra,funkcja_pietra,liczba_kondygnacji,mieszana,rok_budowy_szac,styl_arch,stan_tech,wysokosc_max,zabytek,zrodlo,pewnosc,data_sprawdzenia
id,,,,,,,,,,,,,,,,,
1,127184291,Świętego Piotra,8,NaN,Hotel,NaN,Hotel,7,<NA>,2000,Współczesność,3,NaN,NaN,google.com/maps,3.0,2026-09-27
2,890729731,Świętego Piotra,6D,NaN,Mieszkaniowa Wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,5,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3.0,2026-09-27
3,890292040,Węglowa,22,NaN,Centrum Handlowe,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3.0,2026-09-27
4,890729732,Świętego Piotra,6,NaN,Mieszkaniowa Wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,Współczesność,3,NaN,NaN,google.com/maps,3.0,2026-09-27
5,136885205,Tadeusza Wendy,15,NaN,Centrum Medyczne,NaN,Centrum Medyczne,6,<NA>,2000,Współczesność,3,NaN,NaN,google.com/maps,2.0,2026-09-27


Data Parsing excercise

In [5]:
data_clean["funkcja_parter"].value_counts().size

380

Number of values in basefloor functions

In [6]:
data_clean["funkcja_parter"] = data_clean["funkcja_parter"].str.lower().str.strip().str.replace("_"," ")
data_clean["funkcja_parter"].value_counts()

funkcja_parter
garaż                                    216
mieszkaniowa wielorodzinna                79
restauracja                               51
nieużytek                                 36
biurowa                                   28
                                        ... 
kawiarnia,cukiernia,usługi podróże         1
żabka,handel odzież,handel epapierosy      1
laboratorium medyczne,piekarnia,kiosk      1
apteka,handel zdrowie                      1
urząd miasta                               1
Name: count, Length: 374, dtype: int64

In [7]:
data_clean["funkcja_parter"].str.count(',').value_counts()

funkcja_parter
0    876
1    150
2     55
3     17
4      4
5      2
7      1
Name: count, dtype: int64

Number of functions per row

In [8]:
mask = data_clean["funkcja_parter"].str.count(',') == 7
data_clean.loc[mask,["funkcja_parter"] ]

,funkcja_parter
id,
959,"piekarnia,żabka,handel medyczny,centrum medycy..."


The id of the building with most functions = 8, based on how many , were in the column

In [9]:
(data_clean["funkcja_parter"].str.count(',')+1).sum()

np.int64(1449)

Number of all functions in the file

In [10]:
data_clean["addr_street"].value_counts()

addr_street
Świętojańska                             125
Antoniego Abrahama                        80
Starowiejska                              59
10 Lutego                                 48
Wójta Radtkego                            45
Władysława IV                             41
Jana z Kolna                              40
Armii Krajowej                            36
Stefana Żeromskiego                       30
Jerzego Waszyngtona                       29
3 Maja                                    26
Hryniewickiego                            22
Skwer Tadeusza Kościuszki                 20
Świętego Piotra                           16
Aleja Jana Pawła II                       15
Plac Kaszubski                            14
Kazimierza Pułaskiego                     14
Jana Kilińskiego                          14
Portowa                                   13
Plac Konstytucji                          11
Zgoda                                     11
Generała Józefa Bema                      1

In [11]:
mask = data_clean["addr_street"] == 'Plac Kaszubski - Jana z Kolna'
data_clean.loc[mask, :]

,osm_id,addr_street,addr_number,grupa_parter,funkcja_parter,grupa_pietra,funkcja_pietra,liczba_kondygnacji,mieszana,rok_budowy_szac,styl_arch,stan_tech,wysokosc_max,zabytek,zrodlo,pewnosc,data_sprawdzenia
id,,,,,,,,,,,,,,,,,
117,127061428,Plac Kaszubski - Jana z Kolna,2,NaN,"hotel,restauracja",NaN,Hotl,5,<NA>,1940,Modernizm,2,NaN,NaN,google.com/maps,3.0,2026-09-27


In [12]:
street_fixes = {
    'Hryniewickiego': 'Antoniego Hryniewickiego',
    'Wendy': 'Tadeusza Wendy',
    'Plac Kaszubski - Jana z Kolna' : 'Jana z Kolna'
}

data_clean["addr_street"] = data_clean["addr_street"].replace(street_fixes)
assert data_clean["addr_street"].nunique() == 45

assert checks if the statement is true, if not it gives an error

In [13]:
data_clean["styl_arch"].unique()

<StringArray>
[  'Współczesność',       'Modernizm',  'Tradycjonalizm',       'Klasycyzm',
         'ArtDeco',   'Współczesnosc', 'Wspóółczesność ',    'Współczsnośc',
   'Traycjonalizm',    'Wspóczesność',       'Art. Deco',     'Nowożytność',
   'Postmodernizm',         'Wiejski',         'Artdeco',     'Wertykalizm',
       'modernizm',     'Nowożytnośc']
Length: 18, dtype: str

In [14]:
data_clean["styl_arch"] = data_clean["styl_arch"].str.strip().str.lower()
data_clean["styl_arch"].unique()

<StringArray>
[ 'współczesność',      'modernizm', 'tradycjonalizm',      'klasycyzm',
        'artdeco',  'współczesnosc', 'wspóółczesność',   'współczsnośc',
  'traycjonalizm',   'wspóczesność',      'art. deco',    'nowożytność',
  'postmodernizm',        'wiejski',    'wertykalizm',    'nowożytnośc']
Length: 16, dtype: str

In [15]:
style_fixes = {
    "nowożytnośc": "współczesność",
    "nowożytność": "współczesność",
    "współczsnośc": "współczesność",
    "wspóółczesność": "współczesność",
    "wspóczesność": "współczesność",
    "współczesnosc": "współczesność",
    "traycjonalizm": "tradycjonalizm",
    "wiejski": "tradycjonalizm",
    "art. deco": "modernizm",
    "artdeco": "modernizm",
    "wertykalizm": "modernizm",
}

data_clean["styl_arch"] = data_clean["styl_arch"].replace(style_fixes)
data_clean["styl_arch"].unique()

<StringArray>
['współczesność', 'modernizm', 'tradycjonalizm', 'klasycyzm', 'postmodernizm']
Length: 5, dtype: str

In [16]:
allowed = {"współczesność", "modernizm", "tradycjonalizm", "klasycyzm", "postmodernizm"}
assert set(data_clean["styl_arch"].unique()) <= allowed

In [17]:
mask = data_clean["rok_budowy_szac"] == 1900
data_clean.loc[mask,:]

,osm_id,addr_street,addr_number,grupa_parter,funkcja_parter,grupa_pietra,funkcja_pietra,liczba_kondygnacji,mieszana,rok_budowy_szac,styl_arch,stan_tech,wysokosc_max,zabytek,zrodlo,pewnosc,data_sprawdzenia
id,,,,,,,,,,,,,,,,,
331,261459260,Starowiejska,30,NaN,restauracja,NaN,NaN,2,<NA>,1900,tradycjonalizm,3,NaN,NaN,google.com/maps,3.0,2026-09-27
339,261459190,Starowiejska,30,NaN,restauracja,NaN,Restauracja,2,<NA>,1900,tradycjonalizm,3,NaN,NaN,google.com/maps,3.0,2026-09-27


In [18]:
data_clean["rok_budowy_szac"].value_counts()

rok_budowy_szac
1980    372
1930    263
1950    205
1940     90
2000     72
2020     42
1920     29
1990     16
2010      5
1970      4
1960      4
1900      2
1910      1
Name: count, dtype: Int64

In [19]:
bins = [1900,1950,1990,2030]
building_cut = pd.cut(data_clean['rok_budowy_szac'], bins, right= False, labels= ['do 1949', '1950–1989', 'od 1990'])
data_clean["okres"] = building_cut

In [20]:
data_clean["addr_number"] = data_clean["addr_number"].str.strip().str.upper()
data_clean["addr_number"].unique()

<StringArray>
[    '8',    '6D',    '22',     '6',    '15',    '19',    '21',    '17',
     '3',    '13',
 ...
   '116',   '103',  '118A',   '57A',   '105',   '118',   '107',   '120',
   '122', '52/54']
Length: 182, dtype: str

In [23]:
pattern1 = r"(\d+)([A-Z]?)"
pattern2 = r"(\d+)[/-](\d+)([A-Z]?)"
mask_pattern1 = data_clean["addr_number"].str.fullmatch(pattern1)
mask_pattern2 = data_clean["addr_number"].str.fullmatch(pattern2)
mask_no_adres =  data_clean["addr_number"].isna()
mask_full = (mask_no_adres) | (mask_pattern1) | (mask_pattern2)
mask_full.sum()
assert mask_full.all()

\d	jedna cyfra	\d pasuje do 7
[A-Z]	jedna litera z zakresu	[A-Z] pasuje do D
+	jedno lub więcej tego, co przed	\d+ pasuje do 7, 12, 104
?	zero lub jedno tego, co przed	[A-Z]? pasuje do litery albo do niczego
( )	grupa, czyli kawałek do wyciągnięcia	(\d+) wyciąga sam numer
r"..."	napis surowy, żeby \ nie znikał	wzorce zawsze pisz jako r"..."